# CYGNSS L1 fixed-operator 2020 arm comparison

Maps and time series comparing the seven fixed-operator CYGNSS L1 assimilation arms staged under
`output/cygl1_fixedop_2020_7arm_stats` (see `README.md` in that directory for full provenance). All
runs share the Arizona limited domain (lon -118..-106, lat 29..40; EASEv2 M36, 909 land tiles) and the
single-year period 2020-01-01 to 2020-12-31, and are all post-fix (GEOSldas.x md5 `6a70b496`).

Each arm is scored two ways in the bundle:

- `DA_paired_<tag>`: the DA arm's own O, F, A.
- `OL_paired_monitor_xmask_<tag>`: the shared open loop's F (A = F), scored against **that arm's**
  scaled obs on the same valid-observation mask (`use_obs=True` cross-mask).

We always compare `DA_paired_<tag>` against `OL_paired_monitor_xmask_<tag>` of the same tag, and report
`100 * (DA - OL) / OL`; **negative = DA has lower OmF residuals than its matching OL (better)**.

| tag | what is assimilated |
|---|---|
| `full_xc015_fixedop` (benchmark) | full CYGNSS L1 stream, xcorr/ycorr 0.15 deg, errstd 2.75 dB |
| `full_xc015_err39_fixedop` | benchmark, errstd 3.9 |
| `full_xc015_coh040216_fixedop` | benchmark + coherency_ratio filter 0.40-2.16 + rebuilt clim |
| `full_xc015_coh040216_err39_fixedop` (best L1 arm) | filter + rebuilt clim + errstd 3.9 |
| `dense075_coh05_fixedop` | thinned L1 (0.75 deg min separation, coherency >= 0.5), xcorr 0.625 deg |
| `smap_fixedop` | SMAP L1C Tb only (L1 monitor-only) |
| `l3_fixedop` | CYGNSS L3 SM only (L1 monitor-only) |

Layout convention (matches `final_dense075_coh05_omf_figures.ipynb`):

- rows/panels group by observing system: **L1, L3, ASCAT, SMOS, SMAP**
- map columns: matching OL, DA, `(DA - OL) / OL * 100`
- negative percent differences mean the DA run has lower OmF residuals than the paired OL baseline


In [ ]:
from __future__ import annotations

import pickle
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize, TwoSlopeNorm
from netCDF4 import Dataset
from IPython.display import display

REPO = Path.cwd()
if REPO.name != "geosldas-analysis":
    REPO = Path("/Users/amfox/Desktop/geosldas-analysis")
PROJECT = REPO / "projects" / "CYGNSS_L1_AZ"
STATS = PROJECT / "output" / "cygl1_fixedop_2020_7arm_stats"
STATSDIR = STATS / "stats"
SUMMARY = STATS / "summary"
OUT = STATS / "figures"
OUT.mkdir(parents=True, exist_ok=True)

MONTH_PERIOD = "202001_202012"
FILE_PERIOD = "20200101_20201231"
PERIOD_LABEL = "2020 (Jan-Dec)"

NMIN = 10
MAP_EXTENT = (-118.6, -105.4, 28.6, 40.4)

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [ ]:
@dataclass(frozen=True)
class Group:
    name: str
    indices: tuple[int, ...]
    units: str
    color: str


# Species axis is identical across every arm and file in this bundle (species_index.csv).
# Names match the `grp` column of summary/noise_gain_*.csv and the headline summary table.
GROUPS = (
    Group("L1", (12,), "dB", "#7b3294"),
    Group("L3", (11,), "m$^3$ m$^{-3}$", "#d95f02"),
    Group("ASCAT", (8, 9, 10), "m$^3$ m$^{-3}$", "#1b9e77"),
    Group("SMOS", (0, 1, 2, 3), "K", "#7570b3"),
    Group("SMAP", (4, 5, 6, 7), "K", "#666666"),
)


@dataclass(frozen=True)
class Arm:
    tag: str
    exp_id: str
    label: str
    note: str


ARMS = (
    Arm("full_xc015_fixedop", "DA_L1_full_xc015_fixedop",
        "Benchmark", "full L1 stream, xcorr/ycorr 0.15 deg, errstd 2.75 dB"),
    Arm("full_xc015_err39_fixedop", "DA_L1_full_xc015_err39_fixedop",
        "Benchmark, errstd 3.9", "benchmark, errstd 3.9"),
    Arm("full_xc015_coh040216_fixedop", "DA_L1_full_xc015_coh040216_fixedop",
        "+ coherency filter", "benchmark + coherency_ratio filter 0.40-2.16 + rebuilt clim"),
    Arm("full_xc015_coh040216_err39_fixedop", "DA_L1_full_xc015_coh040216_err39_fixedop",
        "Best L1 arm", "filter + rebuilt clim + errstd 3.9"),
    Arm("dense075_coh05_fixedop", "DA_L1_dense075_coh05_fixedop",
        "Thinned (dense075/coh05)", "0.75 deg min separation, coherency >= 0.5, xcorr 0.625 deg"),
    Arm("smap_fixedop", "DA_SMAP_fixedop",
        "SMAP-only", "SMAP L1C Tb only (L1 monitor-only)"),
    Arm("l3_fixedop", "DA_L3_fixedop",
        "L3-only", "CYGNSS L3 SM only (L1 monitor-only)"),
)
ARM_BY_TAG = {arm.tag: arm for arm in ARMS}
ARM_COLORS = {
    "full_xc015_fixedop": "#4c72b0",
    "full_xc015_err39_fixedop": "#55a868",
    "full_xc015_coh040216_fixedop": "#c44e52",
    "full_xc015_coh040216_err39_fixedop": "#8172b2",
    "dense075_coh05_fixedop": "#ccb974",
    "smap_fixedop": "#64b5cd",
    "l3_fixedop": "#e17c05",
}

BENCHMARK_TAG = "full_xc015_fixedop"
BEST_TAG = "full_xc015_coh040216_err39_fixedop"
FOCUS_TAGS = (BENCHMARK_TAG, BEST_TAG)


def arm_files(tag: str) -> dict[str, Path]:
    return {
        "ol_nc4": STATSDIR / f"temporal_stats_OL_paired_monitor_xmask_{tag}_{FILE_PERIOD}.nc4",
        "da_nc4": STATSDIR / f"temporal_stats_DA_paired_{tag}_{FILE_PERIOD}.nc4",
        "ol_pkl": STATSDIR / f"spatial_stats_OL_paired_monitor_xmask_{tag}_{MONTH_PERIOD}.pkl",
        "da_pkl": STATSDIR / f"spatial_stats_DA_paired_{tag}_{MONTH_PERIOD}.pkl",
    }


missing = [
    (arm.tag, key, path)
    for arm in ARMS
    for key, path in arm_files(arm.tag).items()
    if not path.exists()
]
print(f"{len(ARMS)} arms, {len(missing)} missing files")
for tag, key, path in missing:
    print("MISSING", tag, key, path)


In [ ]:
def read_nc(path: Path, names=("OmF_stdv", "OmF_mean", "O_mean", "F_mean", "N_data")) -> dict[str, np.ndarray]:
    with Dataset(path) as ds:
        return {
            name: np.ma.filled(ds.variables[name][:], np.nan).astype(float)
            for name in names
        }


def read_monthly(path: Path) -> dict[str, np.ndarray]:
    with path.open("rb") as stream:
        raw = pickle.load(stream)
    data = {
        key: (np.asarray(value, dtype=float) if key != "date_vec" else value)
        for key, value in raw.items()
    }
    data["months"] = np.asarray([datetime.strptime(str(v)[:6], "%Y%m") for v in data.get("date_vec", [])])
    return data


def weighted_species(values: np.ndarray, counts: np.ndarray, group: Group, nmin: int = NMIN) -> np.ndarray:
    """N-weighted average of a variable across the species that belong to one observing system."""
    idx = list(group.indices)
    group_vals = values[..., idx]
    group_counts = counts[..., idx]
    valid = np.isfinite(group_vals) & np.isfinite(group_counts) & (group_counts >= nmin)
    numerator = np.where(valid, group_vals * group_counts, 0.0).sum(axis=-1)
    denominator = np.where(valid, group_counts, 0.0).sum(axis=-1)
    return np.divide(numerator, denominator, out=np.full(denominator.shape, np.nan), where=denominator > 0)


def group_values(data: dict[str, np.ndarray], variable: str, group: Group) -> np.ndarray:
    return weighted_species(data[variable], data["N_data"], group)


def group_weight(data: dict[str, np.ndarray], group: Group) -> np.ndarray:
    return np.nansum(data["N_data"][..., list(group.indices)], axis=-1)


def percent_diff(da: np.ndarray, ol: np.ndarray) -> np.ndarray:
    return 100.0 * (da - ol) / ol


def n_weighted_mean(values: np.ndarray, weight: np.ndarray) -> float:
    valid = np.isfinite(values) & np.isfinite(weight) & (weight > 0)
    if not np.any(valid):
        return np.nan
    return float(np.average(values[valid], weights=weight[valid]))


def norm_from_values(values, lower=2, upper=98):
    finite_parts = [np.ravel(v[np.isfinite(v)]) for v in values if np.any(np.isfinite(v))]
    if not finite_parts:
        return Normalize(0, 1)
    finite = np.concatenate(finite_parts)
    vmin, vmax = np.nanpercentile(finite, (lower, upper))
    if np.isclose(vmin, vmax):
        delta = max(abs(vmin) * 0.05, 1.0e-6)
        vmin -= delta
        vmax += delta
    return Normalize(vmin=vmin, vmax=vmax)


def signed_norm(values, limit=None, percentile=98):
    if limit is None:
        finite_parts = [np.ravel(np.abs(v[np.isfinite(v)])) for v in values if np.any(np.isfinite(v))]
        finite = np.concatenate(finite_parts) if finite_parts else np.asarray([1.0])
        limit = float(np.nanpercentile(finite, percentile))
    limit = max(float(limit), 1.0e-12)
    return TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)


def base_map(ax):
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.94", zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor="white", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.45, edgecolor="0.25")
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.35, edgecolor="0.35")
    states = cfeature.NaturalEarthFeature(
        "cultural", "admin_1_states_provinces_lakes", "50m", facecolor="none"
    )
    ax.add_feature(states, lw=0.35, edgecolor="0.45")


def scatter_tiles(ax, values, cmap, norm):
    valid = np.isfinite(values)
    return ax.scatter(
        lon[valid], lat[valid], c=values[valid], s=19, marker="s", linewidths=0,
        cmap=cmap, norm=norm, transform=ccrs.PlateCarree(), zorder=2,
    )


def setup_month_axis(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.grid(True, lw=0.4, alpha=0.35)


In [ ]:
tile_coords = pd.read_csv(STATS / "tile_coords.csv").sort_values("tile_axis_index").reset_index(drop=True)
assert (tile_coords["tile_axis_index"].to_numpy() == np.arange(len(tile_coords))).all(), \
    "tile_coords.csv rows must already be in tile-axis order"
lon = tile_coords["com_lon"].to_numpy(dtype=float)
lat = tile_coords["com_lat"].to_numpy(dtype=float)

species_index = pd.read_csv(STATS / "species_index.csv")
for group in GROUPS:
    descrs = species_index.loc[list(group.indices), "descr"].tolist()
    print(f"{group.name:6s} idx={group.indices}  {descrs}")

print(f"\n{len(tile_coords)} tiles, lon [{lon.min():.2f}, {lon.max():.2f}], lat [{lat.min():.2f}, {lat.max():.2f}]")


In [ ]:
DATA = {}
for arm in ARMS:
    files = arm_files(arm.tag)
    DATA[arm.tag] = {
        "temporal": {"ol": read_nc(files["ol_nc4"]), "da": read_nc(files["da_nc4"])},
        "monthly": {"ol": read_monthly(files["ol_pkl"]), "da": read_monthly(files["da_pkl"])},
    }

for arm in ARMS:
    n_tile = DATA[arm.tag]["temporal"]["da"]["OmF_stdv"].shape[0]
    n_month = DATA[arm.tag]["monthly"]["da"]["OmF_stdv"].shape[0]
    print(f"{arm.tag:38s} tiles={n_tile:4d}  months={n_month:2d}  ({arm.label})")


## Headline: full-year OmF standard-deviation change vs matching OL

This reproduces `summary/month_arm_omf_stdv_pct_vs_OL.csv` (produced by the toolkit's own
`build_month_arm_table.py`), which pools each species' monthly OmF stdv across the full year
(N-weighted) before taking `100 * (DA - OL) / OL`, then averages that percent change across the
species in each observing system. Negative = DA better. This is the same table quoted in the
bundle's `README.md`.


In [ ]:
headline = pd.read_csv(SUMMARY / "month_arm_omf_stdv_pct_vs_OL.csv")
group_cols = [g.name for g in GROUPS]

headline_year = (
    headline[headline["period"] == 2020]
    .set_index("arm")
    .loc[[arm.tag for arm in ARMS], group_cols + ["N_L1"]]
)
headline_year.index = [ARM_BY_TAG[tag].label for tag in headline_year.index]
display(headline_year.style.format({c: "{:+.2f}" for c in group_cols}).format({"N_L1": "{:.0f}"}))

fig, (ax_heat, ax_bar) = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"width_ratios": [1.4, 1]})

matrix = headline_year[group_cols].to_numpy()
heat_norm = signed_norm([matrix])
image = ax_heat.imshow(matrix, cmap="RdBu_r", norm=heat_norm, aspect="auto")
ax_heat.set_xticks(range(len(group_cols)))
ax_heat.set_xticklabels(group_cols)
ax_heat.set_yticks(range(len(headline_year)))
ax_heat.set_yticklabels(headline_year.index, fontsize=8.5)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax_heat.text(j, i, f"{matrix[i, j]:+.2f}", ha="center", va="center", fontsize=8)
ax_heat.set_title("Full-year OmF StDev % change vs matching OL", fontweight="bold")
fig.colorbar(image, ax=ax_heat, orientation="vertical", fraction=0.05, pad=0.02, label="%")

bar_labels = headline_year.index
bar_colors = [ARM_COLORS[tag] for tag in [arm.tag for arm in ARMS]]
ax_bar.barh(range(len(bar_labels)), headline_year["N_L1"], color=bar_colors)
ax_bar.set_yticks(range(len(bar_labels)))
ax_bar.set_yticklabels(bar_labels, fontsize=8.5)
ax_bar.invert_yaxis()
ax_bar.set_xlabel("N obs (L1 species, full year)")
ax_bar.set_title("CYGNSS L1 obs population by arm", fontweight="bold")

fig.suptitle(f"Headline summary, {PERIOD_LABEL}", y=1.03, fontsize=13)
fig.tight_layout()
headline_path = OUT / "headline_omf_stdv_pct_heatmap.png"
fig.savefig(headline_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", headline_path.relative_to(PROJECT))


## Does the domain-wide comparison include tiles L1 never observes?

Every percent-change number above is a weighted mean over each *species'* own valid-tile footprint —
but the sensors do not share the same footprint within this 909-tile AZ box. CYGNSS L1 is not
global-coverage: its orbital inclination limits it to roughly +/-38 degrees latitude, and this domain
runs from 29 to 40 degrees North. Before trusting a domain-wide SMOS/SMAP number for an L1 arm, it's
worth checking how much of the domain L1 actually reaches, and whether comparisons against the
monitor-only arms (which have their own, only partially-overlapping footprints) are being run over
tiles where L1 could plausibly have had any effect at all.

(This is the same idea as the `LAT_BOUNDARY = 37.5` restriction in the original
`final_dense075_coh05_omf_figures.ipynb` notebook, which annotated every map with the area-weighted mean
*south of 37.5N* rather than the full-domain mean, for exactly this reason.)


In [ ]:
best_da_temporal = DATA[BEST_TAG]["temporal"]["da"]
footprint_rows = []
for i, row in species_index.iterrows():
    n = best_da_temporal["N_data"][:, i]
    group_name = next((g.name for g in GROUPS if i in g.indices), None)
    footprint_rows.append({"species": row["descr"], "group": group_name, "tiles_with_obs": int(np.sum(n > 0))})
species_footprint = pd.DataFrame(footprint_rows)
display(species_footprint)

group_footprint_rows = []
for group in GROUPS:
    any_obs = np.any(best_da_temporal["N_data"][:, list(group.indices)] > 0, axis=-1)
    group_footprint_rows.append({
        "group": group.name, "tiles_with_obs": int(any_obs.sum()),
        "pct_of_domain": 100 * any_obs.sum() / len(any_obs),
    })
print(f"\n{len(tile_coords)}-tile domain, coverage by system ({ARM_BY_TAG[BEST_TAG].label}):")
display(pd.DataFrame(group_footprint_rows).style.format({"pct_of_domain": "{:.0f}%"}))

# CYGNSS L1's own footprint: the mask used to restrict the comparison below.
L1_FOOTPRINT = best_da_temporal["N_data"][:, 12] > 0
print(f"\nL1 footprint: {L1_FOOTPRINT.sum()} of {len(L1_FOOTPRINT)} tiles ({100 * L1_FOOTPRINT.mean():.0f}%)")

lat_bins = np.arange(29, 41, 1)
print("\nL1 coverage by 1-degree latitude band:")
for b0, b1 in zip(lat_bins[:-1], lat_bins[1:]):
    band = (lat >= b0) & (lat < b1)
    n_band, n_l1 = band.sum(), (band & L1_FOOTPRINT).sum()
    print(f"  [{b0:.0f}, {b1:.0f}) N: {n_band:3d} tiles, {n_l1:3d} with L1 obs ({100 * n_l1 / max(n_band, 1):.0f}%)")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.8), subplot_kw={"projection": ccrs.PlateCarree()})
base_map(ax)
ax.scatter(lon[~L1_FOOTPRINT], lat[~L1_FOOTPRINT], c="0.75", s=19, marker="s", linewidths=0,
           transform=ccrs.PlateCarree(), zorder=2, label=f"no L1 obs ({int((~L1_FOOTPRINT).sum())} tiles)")
ax.scatter(lon[L1_FOOTPRINT], lat[L1_FOOTPRINT], c="#7b3294", s=19, marker="s", linewidths=0,
           transform=ccrs.PlateCarree(), zorder=2, label=f"has L1 obs ({int(L1_FOOTPRINT.sum())} tiles)")
ax.legend(loc="lower left", fontsize=8, framealpha=0.9)
ax.set_title(
    f"CYGNSS L1 observational footprint, {PERIOD_LABEL}\n"
    f"{L1_FOOTPRINT.sum()} of {len(L1_FOOTPRINT)} tiles ({100 * L1_FOOTPRINT.mean():.0f}%) "
    "-- note the cutoff above ~37.4N (orbital inclination)",
    fontweight="bold", fontsize=10,
)
footprint_path = OUT / "l1_observational_footprint_map.png"
fig.savefig(footprint_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", footprint_path.relative_to(PROJECT))


### All seven arms, restricted to the region L1 actually observes

Repeats the annual per-tile domain mean for every arm, restricting the SMOS/SMAP/ASCAT/L3 tiles to only
the ones where CYGNSS L1 has an observation (the same mask for every arm, including the two monitor-only
arms, so this is an apples-to-apples "what happens in the part of the domain L1 could influence" view).


In [ ]:
footprint_comparison_rows = []
for arm in ARMS:
    temporal = DATA[arm.tag]["temporal"]
    for group in GROUPS:
        ol = group_values(temporal["ol"], "OmF_stdv", group)
        da = group_values(temporal["da"], "OmF_stdv", group)
        pct = percent_diff(da, ol)
        weight = group_weight(temporal["da"], group)
        footprint_comparison_rows.append({
            "arm": arm.label,
            "system": group.name,
            "full_domain_pct": n_weighted_mean(pct, weight),
            "l1_footprint_only_pct": n_weighted_mean(pct, weight * L1_FOOTPRINT),
        })

footprint_comparison = pd.DataFrame(footprint_comparison_rows)
pivot_full = footprint_comparison.pivot(index="arm", columns="system", values="full_domain_pct").loc[[a.label for a in ARMS], [g.name for g in GROUPS]]
pivot_restricted = footprint_comparison.pivot(index="arm", columns="system", values="l1_footprint_only_pct").loc[[a.label for a in ARMS], [g.name for g in GROUPS]]

fig, (ax_full, ax_restr) = plt.subplots(1, 2, figsize=(13.5, 4.6), sharey=True)
shared_norm = signed_norm([pivot_full.to_numpy(), pivot_restricted.to_numpy()])
for ax, pivot, title in ((ax_full, pivot_full, "Full domain (each system's own footprint)"),
                          (ax_restr, pivot_restricted, "Restricted to the L1 footprint (563 tiles)")):
    image = ax.imshow(pivot.to_numpy(), cmap="RdBu_r", norm=shared_norm, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.iat[i, j]:+.2f}", ha="center", va="center", fontsize=7.5)
    ax.set_title(title, fontweight="bold", fontsize=10)
fig.colorbar(image, ax=[ax_full, ax_restr], orientation="vertical", fraction=0.03, pad=0.02, label="%")
fig.suptitle("OmF StDev % change vs matching OL: full domain vs L1-footprint-restricted", y=1.02, fontsize=12)
footprint_comparison_path = OUT / "all_arms_footprint_restricted_comparison.csv"
footprint_comparison.to_csv(footprint_comparison_path, index=False)
footprint_fig_path = OUT / "all_arms_footprint_restricted_heatmaps.png"
fig.savefig(footprint_fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", footprint_fig_path.relative_to(PROJECT))
print("wrote", footprint_comparison_path.relative_to(PROJECT))


## Cross-arm monthly time series (percent change vs matching OL)

Same source table as above, at monthly resolution: one panel per observing system, one line per arm.


In [ ]:
headline_monthly = headline[headline["period"] != 2020].copy()
headline_monthly["month"] = pd.to_datetime(headline_monthly["period"].astype(str), format="%Y%m")

fig, axes = plt.subplots(len(GROUPS), 1, figsize=(10.5, 12.0), sharex=True)
for ax, group in zip(axes, GROUPS):
    ax.axhline(0, color="0.25", lw=0.8, zorder=0)
    for arm in ARMS:
        rows = headline_monthly[headline_monthly["arm"] == arm.tag].sort_values("month")
        ax.plot(rows["month"], rows[group.name], "-o", color=ARM_COLORS[arm.tag],
                lw=1.7, ms=3.2, label=arm.label)
    ax.set_ylabel(f"{group.name}\n%", fontweight="bold")
    setup_month_axis(ax)

axes[-1].set_xlabel("month")
for tick in axes[-1].get_xticklabels():
    tick.set_rotation(35)
    tick.set_ha("right")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.99), fontsize=8.5)
fig.suptitle(
    f"Monthly OmF StDev change vs matching OL, all arms, {PERIOD_LABEL}\n"
    "100 x (DA - OL) / OL; negative = lower DA residual",
    y=1.05, fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.93))
monthly_pct_path = OUT / "cross_arm_monthly_omf_stdv_percent_5x1.png"
fig.savefig(monthly_pct_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", monthly_pct_path.relative_to(PROJECT))


## Per-tile annual maps: benchmark and best L1 arm

Full OL / DA / percent-difference maps (from the per-tile `temporal_stats_*.nc4` files) for the two
arms called out in the README: the `full_xc015` benchmark and the best-performing L1 arm
(`full_xc015_coh040216_err39`).


In [ ]:
def plot_arm_omf_stdv_maps(tag: str):
    arm = ARM_BY_TAG[tag]
    temporal = DATA[tag]["temporal"]

    rows = []
    for group in GROUPS:
        ol = group_values(temporal["ol"], "OmF_stdv", group)
        da = group_values(temporal["da"], "OmF_stdv", group)
        pct = percent_diff(da, ol)
        rows.append((group, ol, da, pct))

    summary_rows = []
    for group, ol, da, pct in rows:
        valid = np.isfinite(pct)
        weight = group_weight(temporal["da"], group)
        summary_rows.append({
            "arm": tag,
            "system": group.name,
            "common_tiles": int(valid.sum()),
            "tile_mean_percent_da_minus_ol": float(np.nanmean(pct)),
            "tile_median_percent_da_minus_ol": float(np.nanmedian(pct)),
            "n_weighted_percent_da_minus_ol": n_weighted_mean(pct, weight),
            "fraction_tiles_improved_da_lt_ol": float(np.mean(pct[valid] < 0)),
        })
    summary = pd.DataFrame(summary_rows)
    summary_path = OUT / f"{tag}_omf_stdv_percent_map_summary.csv"
    summary.to_csv(summary_path, index=False)

    fig = plt.figure(figsize=(13.2, 14.8))
    percent_norm = signed_norm([row[3] for row in rows])
    for row_index, (group, ol, da, pct) in enumerate(rows):
        value_norm = norm_from_values([ol, da])
        for col_index, (values, title, cmap, norm) in enumerate((
            (ol, "OL OmF StDev", "viridis", value_norm),
            (da, "DA OmF StDev", "viridis", value_norm),
            (pct, "(DA - OL) / OL (%)", "RdBu_r", percent_norm),
        )):
            ax = fig.add_subplot(5, 3, row_index * 3 + col_index + 1, projection=ccrs.PlateCarree())
            base_map(ax)
            image = scatter_tiles(ax, values, cmap, norm)
            if row_index == 0:
                ax.set_title(title, fontweight="bold", pad=8)
            if col_index == 0:
                ax.text(-0.07, 0.5, group.name, transform=ax.transAxes, rotation=90,
                        va="center", ha="center", fontweight="bold", fontsize=11)
            if col_index == 2:
                mean_value = summary.loc[summary.system == group.name, "n_weighted_percent_da_minus_ol"].iloc[0]
                ax.text(0.02, 0.03, f"N-wtd mean: {mean_value:+.2f}%",
                        transform=ax.transAxes, fontsize=8,
                        bbox={"facecolor": "white", "edgecolor": "0.7", "alpha": 0.88, "pad": 2})
            ax.set_xticks([])
            ax.set_yticks([])
            label = "%" if col_index == 2 else group.units
            cb = fig.colorbar(image, ax=ax, orientation="horizontal", fraction=0.046, pad=0.035)
            cb.set_label(label, fontsize=8)
            cb.ax.tick_params(labelsize=7)

    fig.suptitle(f"{arm.label} ({tag}): OmF standard deviation maps, {PERIOD_LABEL}", y=0.995, fontsize=14)
    fig.subplots_adjust(top=0.965, bottom=0.025, left=0.055, right=0.985, hspace=0.25, wspace=0.08)
    map_path = OUT / f"{tag}_omf_stdv_maps_5x3.png"
    fig.savefig(map_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("wrote", map_path.relative_to(PROJECT))
    print("wrote", summary_path.relative_to(PROJECT))
    return summary


focus_map_summaries = {}
for tag in FOCUS_TAGS:
    focus_map_summaries[tag] = plot_arm_omf_stdv_maps(tag)
    display(focus_map_summaries[tag])


## Cross-arm comparison: annual percent-difference maps, all seven arms

Rows = arms, columns = observing systems. A single shared diverging color scale (98th percentile of
the magnitude across every arm and system) makes it possible to compare the L1 arms against the two
monitor-only arms (`smap`, `l3`) on one figure; the monitor-only arms genuinely have much larger swings
in the system they assimilate, so they will look saturated relative to the L1 arms.


In [ ]:
combined_rows = []
combined_summary_rows = []
for arm in ARMS:
    temporal = DATA[arm.tag]["temporal"]
    row = []
    for group in GROUPS:
        ol = group_values(temporal["ol"], "OmF_stdv", group)
        da = group_values(temporal["da"], "OmF_stdv", group)
        pct = percent_diff(da, ol)
        weight = group_weight(temporal["da"], group)
        mean_value = n_weighted_mean(pct, weight)
        row.append((group, pct, mean_value))
        combined_summary_rows.append({
            "arm": arm.tag,
            "system": group.name,
            "common_tiles": int(np.isfinite(pct).sum()),
            "n_weighted_percent_da_minus_ol": mean_value,
        })
    combined_rows.append((arm, row))

combined_summary = pd.DataFrame(combined_summary_rows)
combined_summary_path = OUT / "cross_arm_omf_stdv_percent_maps_7x5_summary.csv"
combined_summary.to_csv(combined_summary_path, index=False)

fig, axes = plt.subplots(
    len(ARMS), len(GROUPS), figsize=(15.5, 18.5),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
combined_norm = signed_norm([pct for _, row in combined_rows for _, pct, _ in row])
for row_index, (arm, row) in enumerate(combined_rows):
    for col_index, (group, pct, mean_value) in enumerate(row):
        ax = axes[row_index, col_index]
        base_map(ax)
        combined_image = scatter_tiles(ax, pct, "RdBu_r", combined_norm)
        if row_index == 0:
            ax.set_title(group.name, fontweight="bold", pad=6)
        if col_index == 0:
            ax.text(-0.14, 0.5, arm.label, transform=ax.transAxes, rotation=90,
                    va="center", ha="center", fontweight="bold", fontsize=9)
        ax.text(0.02, 0.03, f"{mean_value:+.2f}%", transform=ax.transAxes, fontsize=7.5,
                bbox={"facecolor": "white", "edgecolor": "0.7", "alpha": 0.9, "pad": 1.5})
        ax.set_xticks([])
        ax.set_yticks([])

colorbar_ax = fig.add_axes((0.2, 0.045, 0.62, 0.012))
colorbar = fig.colorbar(combined_image, cax=colorbar_ax, orientation="horizontal")
colorbar.set_label(
    "(DA - OL) / OL OmF StDev (%); negative = lower DA residual; annotations are N-weighted domain means",
    fontsize=9,
)
colorbar.ax.tick_params(labelsize=8)
fig.suptitle(
    f"OmF standard deviation change vs matching OL, all seven arms, {PERIOD_LABEL}",
    y=0.995, fontsize=14,
)
fig.subplots_adjust(top=0.965, bottom=0.075, left=0.09, right=0.99, hspace=0.12, wspace=0.08)
combined_map_path = OUT / "cross_arm_omf_stdv_percent_maps_7x5.png"
fig.savefig(combined_map_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", combined_map_path.relative_to(PROJECT))
print("wrote", combined_summary_path.relative_to(PROJECT))


## Per-tile monthly time series: benchmark and best L1 arm

Domain-wide monthly OmF StDev (absolute values, N-weighted across species in each system) for OL vs
DA, plus the percent difference, from the per-month `spatial_stats_*.pkl` files. Shown in native units
so the size of the OL/DA gap is visible, not just its sign.


In [ ]:
def plot_arm_monthly_omf_stdv(tag: str):
    arm = ARM_BY_TAG[tag]
    monthly = DATA[tag]["monthly"]
    months = monthly["ol"]["months"]
    assert np.array_equal(months, monthly["da"]["months"]), f"month vectors differ for {tag}"

    fig, axes = plt.subplots(5, 2, figsize=(13.0, 12.6), sharex=True)
    summary_rows = []
    for row_index, group in enumerate(GROUPS):
        left, right = axes[row_index, 0], axes[row_index, 1]
        ol = group_values(monthly["ol"], "OmF_stdv", group)
        da = group_values(monthly["da"], "OmF_stdv", group)
        pct = percent_diff(da, ol)

        left.plot(months, ol, "--o", color="0.35", lw=1.45, ms=3.3, label="matching OL" if row_index == 0 else None)
        left.plot(months, da, "-o", color=group.color, lw=1.9, ms=3.6, label="DA" if row_index == 0 else None)
        right.plot(months, pct, "-o", color=group.color, lw=1.9, ms=3.6)
        right.axhline(0, color="0.25", lw=0.8)

        left.set_ylabel(f"{group.name}\n{group.units}", fontweight="bold")
        right.set_ylabel("%")
        setup_month_axis(left)
        setup_month_axis(right)
        if row_index == 0:
            left.set_title("OmF StDev: matching OL and DA", fontweight="bold")
            right.set_title("Percent difference from matching OL", fontweight="bold")
        if row_index == len(GROUPS) - 1:
            left.set_xlabel("month")
            right.set_xlabel("month")
        for axis in (left, right):
            for label in axis.get_xticklabels():
                label.set_rotation(35)
                label.set_ha("right")

        valid = np.isfinite(pct)
        summary_rows.append({
            "arm": tag,
            "system": group.name,
            "monthly_mean_percent_da_minus_ol": float(np.nanmean(pct)),
            "months_improved_da_lt_ol": int(np.sum(pct[valid] < 0)),
            "months_valid": int(valid.sum()),
        })

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.985))
    fig.suptitle(
        f"{arm.label} ({tag}): monthly OmF StDev, {PERIOD_LABEL}\n"
        "Percent difference uses 100 x (DA - OL) / OL; negative = lower DA residual",
        y=1.025, fontsize=14,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    path = OUT / f"{tag}_monthly_omf_stdv_5x2.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()

    table = pd.DataFrame(summary_rows)
    table_path = OUT / f"{tag}_monthly_omf_stdv_summary.csv"
    table.to_csv(table_path, index=False)
    print("wrote", path.relative_to(PROJECT))
    print("wrote", table_path.relative_to(PROJECT))
    return table


for tag in FOCUS_TAGS:
    display(plot_arm_monthly_omf_stdv(tag))


## Noise vs. gain split (MSE decomposition)

From `summary/noise_gain_<tag>_*.csv` (computed directly from the ens_avg ObsFcstAna files, matched
DA-to-OL on time/species/tile). With `inn = O - F_OL` and `dF = F_DA - F_OL`:

- `dMSE = noise + gain`, all as % of the OL MSE (`noise = sum(dF^2)`, `gain = -2 * sum(dF * inn)`)
- `corr = corr(dF, inn)`, `alpha_opt` = increment scaling that would minimize MSE

This is a different decomposition than the OmF-stdv percent change above (it includes the O-F bias),
and the README's headline notes that in every L1 arm, May-June Tb gets worse because `corr(dF, inn)`
collapses toward zero for SMAP/SMOS in those months (a signal problem, not an R-amplitude problem).


In [ ]:
NOISE_GAIN = {}
for arm in ARMS:
    path = SUMMARY / f"noise_gain_{arm.tag}_{MONTH_PERIOD}.csv"
    df = pd.read_csv(path)
    df["month_dt"] = pd.to_datetime(df["month"], format="%Y-%m", errors="coerce")
    NOISE_GAIN[arm.tag] = df

fig, axes = plt.subplots(len(GROUPS), 1, figsize=(10.5, 12.0), sharex=True)
for ax, group in zip(axes, GROUPS):
    ax.axhline(0, color="0.25", lw=0.8, zorder=0)
    for arm in ARMS:
        rows = NOISE_GAIN[arm.tag]
        monthly = rows[(rows["grp"] == group.name) & rows["month_dt"].notna()].sort_values("month_dt")
        ax.plot(monthly["month_dt"], monthly["dMSE"], "-o", color=ARM_COLORS[arm.tag],
                lw=1.7, ms=3.2, label=arm.label)
    ax.set_ylabel(f"{group.name}\ndMSE %", fontweight="bold")
    setup_month_axis(ax)

axes[-1].set_xlabel("month")
for tick in axes[-1].get_xticklabels():
    tick.set_rotation(35)
    tick.set_ha("right")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.99), fontsize=8.5)
fig.suptitle(
    f"Monthly MSE change vs matching OL (noise + gain decomposition), {PERIOD_LABEL}\n"
    "dMSE as % of the OL MSE; negative = DA reduces MSE",
    y=1.05, fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.93))
dmse_path = OUT / "cross_arm_monthly_dmse_percent_5x1.png"
fig.savefig(dmse_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", dmse_path.relative_to(PROJECT))


In [ ]:
full_year_noise_gain = pd.concat(
    [NOISE_GAIN[arm.tag][NOISE_GAIN[arm.tag]["month"] == "all"].assign(arm=arm.tag) for arm in ARMS],
    ignore_index=True,
)
full_year_pivot = full_year_noise_gain.pivot(index="arm", columns="grp", values="corr").loc[
    [arm.tag for arm in ARMS], [g.name for g in GROUPS]
]
full_year_pivot.index = [ARM_BY_TAG[tag].label for tag in full_year_pivot.index]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
image = ax.imshow(full_year_pivot.to_numpy(), cmap="viridis", vmin=0, vmax=full_year_pivot.to_numpy().max())
ax.set_xticks(range(len(GROUPS)))
ax.set_xticklabels([g.name for g in GROUPS])
ax.set_yticks(range(len(full_year_pivot)))
ax.set_yticklabels(full_year_pivot.index, fontsize=8.5)
for i in range(full_year_pivot.shape[0]):
    for j in range(full_year_pivot.shape[1]):
        ax.text(j, i, f"{full_year_pivot.iat[i, j]:.2f}", ha="center", va="center", fontsize=8,
                 color="white" if full_year_pivot.iat[i, j] < full_year_pivot.to_numpy().max() * 0.6 else "black")
ax.set_title(f"Full-year corr(dF, inn), {PERIOD_LABEL}\nlow corr = increments uncorrelated with O-F(OL), i.e. a weak/noisy signal", fontweight="bold", fontsize=10)
fig.colorbar(image, ax=ax, fraction=0.05, pad=0.02, label="corr(dF, inn)")
fig.tight_layout()
corr_path = OUT / "cross_arm_fullyear_corr_heatmap.png"
fig.savefig(corr_path, dpi=180, bbox_inches="tight")
plt.show()

noise_gain_summary_path = OUT / "cross_arm_fullyear_noise_gain_summary.csv"
full_year_noise_gain.to_csv(noise_gain_summary_path, index=False)
display(full_year_noise_gain[["arm", "grp", "N", "dMSE", "noise", "gain", "corr", "alpha_opt"]])
print("wrote", corr_path.relative_to(PROJECT))
print("wrote", noise_gain_summary_path.relative_to(PROJECT))


Saved outputs (all under `output/cygl1_fixedop_2020_7arm_stats/figures/`):

- `headline_omf_stdv_pct_heatmap.png`
- `cross_arm_monthly_omf_stdv_percent_5x1.png`
- `<benchmark_tag>_omf_stdv_maps_5x3.png`, `<best_tag>_omf_stdv_maps_5x3.png` (+ `*_summary.csv`)
- `cross_arm_omf_stdv_percent_maps_7x5.png` (+ `*_summary.csv`)
- `<benchmark_tag>_monthly_omf_stdv_5x2.png`, `<best_tag>_monthly_omf_stdv_5x2.png` (+ `*_summary.csv`)
- `cross_arm_monthly_dmse_percent_5x1.png`
- `cross_arm_fullyear_corr_heatmap.png`
- `cross_arm_fullyear_noise_gain_summary.csv`

For the full write-up and month x arm tables behind this bundle, see
`cygl1_fixedop_2020_arm_comparison_report.md` in the same directory.
